# Analyse des Risques Environnementaux - Réseau Ferroviaire SNCF

Ce projet réalise une analyse quantitative des risques environnementaux impactant le Réseau Ferré National (RFN). 

**Objectif :** Identifier les tronçons vulnérables, quantifier le risque et proposer des stratégies d'adaptation.

**Sources de données :**
- Géométrie et Vitesses : API SNCF Open Data
- Risques Inondation : [Géorisques](https://www.georisques.gouv.fr) / [data.gouv.fr](https://www.data.gouv.fr)
- Biodiversité : [Natura 2000](https://natura2000.europa.eu/)

## 1. Visualisation du Réseau Ferroviaire
L'objectif est d'abord de cartographier l'ensemble du réseau pour disposer d'une base géométrique fiable.

In [ ]:
import folium
import requests
import pandas as pd
import logging

# 1. Initialisation de la Carte
sncf_map = folium.Map(
    location=[46.603354, 1.888334], zoom_start=6, tiles='CartoDB positron', height=800
)

URL_SHAPES = "https://ressources.data.sncf.com/api/explore/v2.1/catalog/datasets/formes-des-lignes-du-rfn/exports/geojson?limit=-1"
URL_SPEED = "https://ressources.data.sncf.com/api/explore/v2.1/catalog/datasets/vitesse-maximale-nominale-sur-ligne/exports/geojson?limit=-1"

def fetch_geojson(url):
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        logging.error(f"Erreur lors de la récupération des données depuis {url}: {e}")
        return None

shapes_data = fetch_geojson(URL_SHAPES)
speed_data = fetch_geojson(URL_SPEED)

if shapes_data and speed_data:
    def get_key(props):
        return f"{props.get('code_ligne')}_{props.get('rg_troncon')}"
    
    final_features = []
    existing_keys = set()
    
    for feature in speed_data.get('features', []):
        props = feature.get('properties', {})
        key = get_key(props)
        existing_keys.add(key)
        standard_props = {
            'Code ligne': props.get('code_ligne', 'Inconnu'),
            'Status': props.get('lib_ligne', props.get('libelle', 'Inconnu')),
            'Vitesse max': props.get('v_max', 'N/A'),
            'Source': 'Vitesse',
        }
        feature['properties'] = {**props, **standard_props}
        final_features.append(feature)
    
    for feature in shapes_data.get('features', []):
        props = feature.get('properties', {})
        key = get_key(props)
        if key not in existing_keys:
            standard_props = {
                'Code ligne': props.get('code_ligne', 'Inconnu'),
                'Status': props.get('libelle', 'Inconnu'),
                'Vitesse max': 'N/A',
                'Source': 'Formes',
            }
            feature['properties'] = {**props, **standard_props}
            final_features.append(feature)
    
    geojson_combined = {'type': 'FeatureCollection', 'features': final_features}
    
    def style_speed(feature):
        vmax = feature['properties'].get('v_max', 'N/A')
        try:
            val = float(vmax)
            if val >= 220:
                color = 'red'
            elif val >= 160:
                color = 'orange'
            elif val >= 80:
                color = 'yellow'
            else:
                color = 'green'
        except (ValueError, TypeError):
            color = 'gray'
        return {'color': color, 'weight': 2, 'opacity': 0.8}
    
    folium.GeoJson(
        geojson_combined,
        name='Réseau SNCF',
        style_function=style_speed,
        tooltip=folium.GeoJsonTooltip(fields=['Code ligne', 'Status', 'Vitesse max']),
    ).add_to(sncf_map)
    folium.LayerControl().add_to(sncf_map)
    logging.info("Carte SNCF générée avec succès.")
else:
    logging.error("Erreur de chargement.")

sncf_map

## 5. Analyse du Risque : Retrait-Gonflement des Argiles

Le retrait-gonflement des argiles est un phénomène naturel qui peut causer des mouvements de terrain et affecter la stabilité des infrastructures ferroviaires. Cette section visualise les zones à risque en France métropolitaine.

In [ ]:
import geopandas as gpd
import folium

# Charger les données de retrait-gonflement des argiles depuis le shapefile
try:
    print("Chargement des données... Cela peut prendre quelques minutes.")
    
    # Lire uniquement les colonnes essentielles pour réduire la mémoire
    all_columns = gpd.read_file('data/AleaRG_2025_Fxx_L93/AleaRG_2025_Fxx_L93.shp').columns.tolist()
    print(f"Colonnes disponibles: {all_columns}")
    
    # Sélectionner uniquement les colonnes nécessaires
    use_columns = ['geometry'] + all_columns[1:4]
    
    # Charger avec seulement les colonnes sélectionnées
    argile_gdf = gpd.read_file('data/AleaRG_2025_Fxx_L93/AleaRG_2025_Fxx_L93.shp', columns=use_columns)
    
    print(f"Données chargées: {len(argile_gdf)} features")
    
    # Simplifier la géométrie
    print("Simplification de la géométrie...")
    argile_gdf['geometry'] = argile_gdf['geometry'].simplify(tolerance=0.0005)
    
    # Créer une nouvelle carte
    argile_map = folium.Map(location=[46.603354, 1.888334], zoom_start=6, tiles='CartoDB positron', height=800)
    
    # Convertir en GeoJSON
    print("Conversion en GeoJSON...")
    argile_geojson = argile_gdf.to_json()
    
    # Ajouter à la carte
    print("Ajout des données à la carte...")
    folium.GeoJson(
        argile_geojson,
        name='Retrait-Gonflement des Argiles',
        style_function=lambda x: {
            'fillColor': 'brown',
            'color': 'brown',
            'weight': 1,
            'fillOpacity': 0.2
        },
        tooltip=folium.GeoJsonTooltip(fields=list(argile_gdf.columns))
    ).add_to(argile_map)
    
    # Ajouter le contrôle des couches
    folium.LayerControl().add_to(argile_map)
    
    print("Carte générée avec succès!")
    argile_map
    
except MemoryError as e:
    print(f"Erreur de mémoire: {e}")
    print("Solutions: utiliser un sous-ensemble ou convertir en GeoJSON")
except Exception as e:
    print(f"Erreur: {e}")
    print("Vérifiez le chemin et la structure des données.")

## 6. Analyse du Risque : Inondations par Ruissellement

Les inondations par ruissellement se produisent lorsque les eaux de pluie ne peuvent pas s'infiltrer dans le sol.

**Source** : [Études d'aléas inondation par ruissellement](https://www.data.gouv.fr/datasets/etudes-daleas-inondation-par-ruissellement)

In [ ]:
import geopandas as gpd
import folium

try:
    print("Chargement des données de ruissellement...")
    ruissellement_gdf = gpd.read_file('data/ruissellement.shp')
    ruissellement_gdf['geometry'] = ruissellement_gdf['geometry'].simplify(tolerance=0.0005)
    
    print(f"Données chargées: {len(ruissellement_gdf)} features")
    
    ruissellement_map = folium.Map(location=[46.603354, 1.888334], zoom_start=6, tiles='CartoDB positron', height=800)
    
    folium.GeoJson(
        ruissellement_gdf.to_json(),
        name='Inondations par Ruissellement',
        style_function=lambda x: {
            'fillColor': 'blue',
            'color': 'blue',
            'weight': 1,
            'fillOpacity': 0.3
        },
        tooltip=folium.GeoJsonTooltip(fields=list(ruissellement_gdf.columns))
    ).add_to(ruissellement_map)
    
    folium.LayerControl().add_to(ruissellement_map)
    print("Carte générée!")
    ruissellement_map
    
except Exception as e:
    print(f"Erreur: {e}")
    print("Téléchargez les données depuis le lien ci-dessus.")

## 7. Analyse du Risque : Inondations par Débordement

Les inondations par débordement de cours d'eau.

**Source** : [Territoires à risque d'inondation (TRI)](https://www.data.gouv.fr/datasets/territoire-a-risque-dinondation-tri-du-sig-directive-inondation-france-metropolitaine-rapportage-2020-100)

In [ ]:
import rasterio
import numpy as np
import folium

try:
    print("Chargement de la carte d'inondation mondiale...")
    
    with rasterio.open('/home/nathan/sncf_risk_assessment/data/floodMapGL_rp100y/floodMapGL_rp100y.tif') as src:
        bounds = src.bounds
        print(f"Carte couvrant: {bounds}")
        
        france_map = folium.Map(location=[46.603354, 1.888334], zoom_start=6, tiles='CartoDB positron', height=800)
        
        fr_left, fr_bottom = 4.8, 41.3
        fr_right, fr_top = 8.2, 51.1
        
        fr_window = rasterio.windows.from_bounds(
            fr_left, fr_bottom, fr_right, fr_top,
            transform=src.transform
        )
        
        folium.Rectangle(
            bounds=[(fr_bottom, fr_left), (fr_top, fr_right)],
            color='blue', fill=True, fill_color='blue', fill_opacity=0.2,
            popup='Zone d\'inondation (scénario 100 ans)'
        ).add_to(france_map)
        
        print("Carte générée!")
        france_map
        
except Exception as e:
    print(f"Erreur: {e}")
    print("Vérifiez le chemin du fichier floodMapGL.")

## 8. Analyse du Risque : Incendies de Forêt

Les incendies de forêt.

**Source** : [Base de Données sur les Incendies de Forêts (BDIFF)](https://www.data.gouv.fr/datasets/base-de-donnees-sur-les-incendies-de-forets-en-france-bdiff)

In [ ]:
import geopandas as gpd
import folium

try:
    print("Chargement des données d'incendie...")
    incendie_gdf = gpd.read_file('data/BDIFF.shp')
    incendie_gdf['geometry'] = incendie_gdf['geometry'].simplify(tolerance=0.0005)
    
    incendie_map = folium.Map(location=[46.603354, 1.888334], zoom_start=6, tiles='CartoDB positron', height=800)
    
    folium.GeoJson(
        incendie_gdf.to_json(),
        name='Zones à Risque d\'Incendie',
        style_function=lambda x: {
            'fillColor': 'red',
            'color': 'red',
            'weight': 1,
            'fillOpacity': 0.3
        },
        tooltip=folium.GeoJsonTooltip(fields=list(incendie_gdf.columns))
    ).add_to(incendie_map)
    
    folium.LayerControl().add_to(incendie_map)
    print("Carte générée!")
    incendie_map
    
except Exception as e:
    print(f"Erreur: {e}")
    print("Téléchargez les données BDIFF.")

## 9. Analyse du Risque : Hausse des Températures

L'augmentation des températures.

**Source** : [Données Publiques de Météo-France](https://donneespubliques.meteofrance.fr/)

In [ ]:
import geopandas as gpd
import folium

try:
    print("Chargement des données de température...")
    temp_gdf = gpd.read_file('data/temperatures.shp')
    
    temp_map = folium.Map(location=[46.603354, 1.888334], zoom_start=6, tiles='CartoDB positron', height=800)
    
    for idx, row in temp_gdf.iterrows():
        hausse = getattr(row, 'hausse_temp', 1)
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=hausse * 3,
            color='orange', fill=True, fill_color='orange', fill_opacity=0.6,
            popup=f"Station: {getattr(row, 'nom', 'Inconnu')}\nHausse: {hausse}°C"
        ).add_to(temp_map)
    
    folium.LayerControl().add_to(temp_map)
    print("Carte générée!")
    temp_map
    
except Exception as e:
    print(f"Erreur: {e}")
    print("Téléchargez les données Météo-France.")